In [1]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from tqdm.notebook import tqdm
from mpl_toolkits.mplot3d import Axes3D
from abc import ABC, abstractmethod
import math
import re
from scipy.optimize import minimize
from itertools import product
import seaborn as sns
from matplotlib import animation
%pip install numba-cuda
import numba as nb
from numba import cuda
%pip install cupy-cuda12x
import cupy as cp

# Apply Periodic Boundary Conditions

In [2]:
def apply_pbc(position, box_size):
    return position % box_size

# Initialize Positions and Velocities

In [12]:
@nb.njit
def apply_pbc(pos: np.ndarray | cp.ndarray | float, box_size: float) -> np.ndarray | cp.ndarray:
    """
    Wraps the chain inside the unit cell
    """
    return pos % box_size

@nb.njit
def initialize_chain_numba(
    n_particles: int,
    box_size: float,
    r0: float,
    rng: np.random._generator.Generator,
    dtype=np.float32
) -> np.ndarray:
    """
    Randomly initialize atom positions by growing the chain
    """
    positions = np.zeros((n_particles, 3), dtype=dtype)

    current_position = np.array([box_size / 2, box_size / 2, box_size / 2], dtype=dtype)
    positions[0] = current_position

    for i in range(1, n_particles):
        direction = rng.normal(size = 3)
        norm = np.sqrt(direction[0]**2 + direction[1]**2 + direction[2]**2)
        direction /= norm

        next_position = current_position + r0 * direction
        positions[i] = apply_pbc(next_position, box_size)
        current_position = positions[i]

    return positions

def initialize_velocities_cupy(
    n_particles: int,
    target_temperature: float, 
    mass: float,
    rng: cp.random._generator_api.Generator,
    kB=1.0,
    dtype=cp.float32
) -> cp.ndarray:
    """
    Initialize particle velocities by drawing from the Maxwell-Botzmann distribution at target temperature
    """
    sigma = cp.sqrt(kB * target_temperature / mass).astype(dtype)

    velocities = rng.standard_normal(
        size=(n_particles, 3), 
        dtype=dtype
    )
    velocities = velocities * sigma

    # Remove center-of-mass velocity
    velocities -= cp.mean(velocities, axis=0)

    return velocities

In [16]:
# qc
rng_np = np.random.default_rng(42)
pos = initialize_chain_numba(10, 20.0, 1.0, rng_np)
rng_cp = cp.random.default_rng(42)
v = initialize_velocities_cupy(1000, target_temperature=1.0, mass=1.0, rng = rng_cp)
v

array([[-0.96734345, -0.23504402, -0.7746753 ],
       [ 1.3575406 , -1.58364   ,  0.3395908 ],
       [ 1.0498317 ,  0.10089839,  1.2355957 ],
       ...,
       [ 0.51645684,  0.2605234 , -0.19178337],
       [-0.09980024,  0.8848019 ,  0.14613646],
       [-0.5017935 ,  0.01541789,  0.9387887 ]], dtype=float32)

# Compute Forces

## Harmonic Forces

In [ ]:
@nb.njit
def minimum_image(displacement, box_size):
    """
    Apply minimum image convention to a 3D displacement vector.
    """
    out = np.empty(3, dtype=displacement.dtype)
    for d in range(3):
        out[d] = displacement[d] - box_size * np.round(displacement[d] / box_size)
    return out

@nb.njit
def compute_harmonic_forces(positions, k, r0, box_size):
    """
    Compute harmonic bond forces for a linear chain.

    Parameters
    ----------
    positions : array, shape (n_particles, 3)
        Particle positions.
    k : float
        Harmonic spring constant.
    r0 : float
        Equilibrium bond length.
    box_size : float
        Periodic box length.

    Returns
    -------
    forces : array, shape (n_particles, 3)
        Force on each particle.
    """
    n_particles = positions.shape[0]
    forces = np.zeros_like(positions)

    for i in range(n_particles - 1):
        displacement = np.empty(3, dtype=positions.dtype)
        for d in range(3):
            displacement[d] = positions[i + 1, d] - positions[i, d]

        displacement = minimum_image(displacement, box_size)

        distance_sq = 0.0
        for d in range(3):
            distance_sq += displacement[d] * displacement[d]
        distance = np.sqrt(distance_sq)

        if distance > 0.0:
            force_magnitude = -k * (distance - r0)

            for d in range(3):
                force_component = force_magnitude * displacement[d] / distance
                forces[i, d] -= force_component
                forces[i + 1, d] += force_component

    return forces

## Lennard-Jones Forces

In [ ]:
@nb.njit
def compute_lennard_jones_forces_njit(
    positions,
    n_particles,
    epsilon_repulsive,
    epsilon_attractive,
    sigma,
    box_size,
    cutoff
):
    forces = np.zeros_like(positions)

    cutoff_sq = cutoff * cutoff

    for i in range(n_particles):
        for j in range(i + 1, n_particles):
            sep = j - i

            if sep == 2:
                epsilon = epsilon_repulsive
            elif sep > 2:
                epsilon = epsilon_attractive
            else:
                continue

            displacement = np.empty(3, dtype=positions.dtype)
            for d in range(3):
                displacement[d] = positions[j, d] - positions[i, d]

            displacement = minimum_image(displacement, box_size)

            r2 = 0.0
            for d in range(3):
                r2 += displacement[d] * displacement[d]

            if r2 < cutoff_sq and r2 > 0.0:
                r = np.sqrt(r2)
                sr = sigma / r
                sr2 = sr * sr
                sr6 = sr2 * sr2 * sr2
                sr12 = sr6 * sr6

                force_magnitude = 24.0 * epsilon * (sr12 - 0.5 * sr6) / r

                for d in range(3):
                    force_component = force_magnitude * displacement[d] / r
                    forces[i, d] -= force_component
                    forces[j, d] += force_component

    return forces

# Velocity Verlet Integration

1. Position update
$$ r(t+\Delta t) = r(t) + v(t)\Delta t + \frac{F(t)}{2m} \Delta t^2 $$
2. Velocity update
$$ v(t+\Delta t) = v(t) + \frac{F(t) + F(t+\Delta t)}{2m} \Delta t $$

Naive implementation
```{python}
for step in range(t_steps):
  forces = compute_forces(positions)  # F(t)
  v += 0.5 * forces / mass * dt
  positions += v * dt
  forces = compute_forces(positions)  # F(t + dt)
  v += 0.5 * forces / mass * dt
  if step % rescale_interval == 0:
    v = rescale(v)
```

Notice that F(t + dt, step i-1) = F(t, step i)
Optimized sequential version
```{python}
acceleration = None
for step in range(t_steps):
  if not acceleration: # first iteration
    forces = compute_forces(positions)  # F(t)
    v += 0.5 * forces / mass * dt
  positions += v * dt
  forces = compute_forces(positions)  # F(t + dt)
  acceleration = forces / mass * dt
  if step % rescale_interval == 0:
    v += 0.5 * acceleration
    v = rescale(v)
    v += 0.5 * acceleration
  else:
    v += acceleration
v -= 0.5 * acceleration
```

compute_forces() needs access to all positions
v, a, f could be stored in registers, while positions need to be stored in global memory
Write the whole thing as one kernel launch, write positions to a trajectory array every save_interval steps

## No coarsening